# Mamba pruning grid - ONE notebook (scores -> fast fine-tuning/eval)
Upload `mamba_grid.zip` to `/content`, edit **cell 1 only**, then **Runtime > Run all**. The notebook works out where it is:

| state | what Run all does |
|---|---|
| no scores yet (fresh runtime) | unit tests -> computes the scores (slow PyTorch path, mamba_ssm must be absent) -> installs the kernel environment -> **restarts the runtime by itself** |
| scores exist, kernels not installed (e.g. a fresh runtime but scores already in Drive) | installs the kernel environment -> restarts the runtime |
| scores exist, kernels installed | runs the fast grid, then packages and downloads the results |

So expect to press **Run all twice** (or once, then again after the automatic restart). Finished runs are skipped, so re-running is always safe.
One GPU session per account at a time: use one account per seed (0 / 1 / 2).

In [ ]:
# ---------------- EDIT THIS CELL ONLY ----------------
# Extras on mamba-130m: layer-0 hybrids, other gate-vs-content pairs, weight-magnitude baselines. Needs the scores from notebook 1 (for hybrids / -CO scores) in Drive; Mag-* need none.
# 21 runs per seed.
# One Colab account per seed (0 / 1 / 2). Upload mamba_grid.zip to /content first (see README: scripts/make_colab_zip.sh).
SEED = 0
MODEL = "state-spaces/mamba-130m-hf"
TAG = ""
METHODS = "Hyb-L0GI,Hyb-L0CO,GxI-CO,Act-CO,GradMag-CO,Mag-GI,Mag-CO"
SPARSITIES = "0.3,0.5,0.7"
SCORE_MICRO = 4
KEEP_MULTIPLE = 8
RUN_TESTS = False
KERNEL_CHECK = False
USE_DRIVE = True
# -------------------------------------------------------


In [ ]:
import os, sys, subprocess, zipfile, glob, shutil
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout: sys.stdout.write(line)
    return p.wait()
OUT = "/content/grid_out" + TAG
if USE_DRIVE:
    from google.colab import drive; drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/mamba_grid_out" + TAG
sh("pip install -q einops scipy datasets")
if not os.path.isdir("/content/MambaLRP"): sh("git clone -q https://github.com/FarnoushRJ/MambaLRP /content/MambaLRP")
sh("pip install -q -e /content/MambaLRP --no-deps")
zipfile.ZipFile("/content/mamba_grid.zip").extractall("/content")      # always re-extract so an old copy is never used
os.chdir("/content/mamba_grid")
sh("nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv")
try:
    import mamba_ssm; FAST_ENV = True
except ImportError:
    FAST_ENV = False
_sp = f"{OUT}/scores/main_seed{SEED}.pt"
_need = set()
for m in METHODS.split(","):
    if m.startswith("Hyb-"): _need |= {"LRP-CO", "LRP-GI"}          # layer-0 hybrids are built from these two cached scores
    elif m not in ("Random", "Dense"): _need.add(m)
HAVE_SCORES = False
if os.path.exists(_sp):
    import torch
    _have = set(torch.load(_sp).keys())
    HAVE_SCORES = _need <= _have                 # a partial file (e.g. pass A saved, pass B crashed) does NOT count
    print("scores in file:", sorted(_have), "| missing:", sorted(_need - _have))
print(f"OUT={OUT}  seed={SEED}  model={MODEL}\nkernels installed: {FAST_ENV}   scores present: {HAVE_SCORES}")

## Orchestrator (scores -> install -> restart -> fast grid)

In [ ]:
if not HAVE_SCORES:
    if FAST_ENV:
        raise SystemExit("Scores are missing but mamba_ssm is installed (scoring needs the pure-PyTorch path). "
                         "Runtime > Disconnect and delete runtime, reconnect, Run all again.")
    if RUN_TESTS:
        assert sh("python test_grid.py /content/MambaLRP && python test_aggregate.py") == 0, "unit tests failed - stop and send me the log"
    rc = sh(f"python mamba_grid.py --stage score --score-micro {SCORE_MICRO} --model {MODEL} --out {OUT} --seeds {SEED} --methods {METHODS} --sparsities {SPARSITIES}")
    assert rc == 0, "scoring failed - send me the log"
    HAVE_SCORES = True

if not FAST_ENV:
    print("\n>>> Installing the kernel environment, then the runtime restarts by itself. Afterwards press Runtime > Run all once more.")
    assert sh("python /content/mamba_grid/install_env.py") == 0, "install_env failed - send me the log"
    os.kill(os.getpid(), 9)          # Colab restarts the runtime; the next Run all continues with the fast grid
else:
    if KERNEL_CHECK:
        sh(f"python kernel_check.py --mambalrp-path /content/MambaLRP")
    rc = sh(f"python mamba_grid.py --stage run --model {MODEL} --out {OUT} --seeds {SEED} --methods {METHODS} --sparsities {SPARSITIES} "
            f"--ft-micro 8 --keep-multiple {KEEP_MULTIPLE}")
    print("grid exit code:", rc)

## Package and download (runs only once kernels are installed)

In [ ]:
if FAST_ENV:
    print(len(glob.glob(f"{OUT}/runs/main/*.json")), "run files in", OUT)
    name = "/content/grid_out%s_seed%d" % (TAG, SEED)
    shutil.make_archive(name, "zip", OUT)
    try:
        from google.colab import files; files.download(name + ".zip")
    except Exception as e: print("download from the Files panel:", e)